# Chapter 9 -- 3D Building Volumetrics, Morphometrics & Microclimate Proxies (Master Class Edition)
## Python for Urban Analysis -- Ghana Edition

---

> **Professor's framing:**
> *"Urban density is three-dimensional. A flat 2D footprint map treats a 10-story hospital the same as a 1-story shed. By computing Gross Floor Area, Volume, Sky View Factor, and Thermal Mass, we bridge spatial data science, architecture, and microclimate physics."*

### Learning Objectives
1. **9.1-9.3**: Multi-tier height ingestion engine & 3D volumetrics ($A$, $H$, $V = A \times H$, $GFA = A \times \text{Floors}$)
2. **9.4**: Hexagonal H3 Res 9 spatial aggregation & Floor Area Ratio ($FAR$)
3. **9.5-9.7**: Height distributions, 2D footprint vs 3D volume comparison, and district skyline rankings
4. **9.8-9.9**: Interactive WebGL 3D Pydeck skyline extrusions & dual-metric visualizations
5. **Prof. Innovation 1**: Sky View Factor (SVF) & Urban Canyon Proxy (Oke, 1988)
6. **Prof. Innovation 2**: Spatial Floor Area Ratio (FAR) & Sprawl Index (Leslie & Martin, 1972)
7. **Prof. Innovation 3**: Volumetric Thermal Mass & UHI Vulnerability (Voogt & Oke, 2003)

## 9.0 -- Install & Configure

In [ ]:
!pip install -q osmnx geopandas h3 pydeck matplotlib contextily mapclassify

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import osmnx as ox
import geopandas as gpd
import pandas as pd
import numpy as np
import h3
import pydeck as pdk
from shapely.geometry import Polygon
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import contextily as cx
from pathlib import Path

STUDY_AREA       = 'Accra, Ghana'
CRS_WEB_MERCATOR = 'EPSG:3857'
CRS_UTM_ACCRA    = 'EPSG:32630'
DARK_BG          = '#0e1117'
BUILDING_TAGS    = {'building': True}

OUTPUT_DIR  = Path('outputs')
FIGURES_DIR = OUTPUT_DIR / 'figures'
for d in [FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def h3_to_polygon(h3_idx):
    boundary_coords = h3.cell_to_boundary(h3_idx)
    return Polygon([(lon, lat) for lat, lon in boundary_coords])

print(f'Study area: {STUDY_AREA}')

## 9.1 -- Fetch & Clean Building Footprints

In [ ]:
print(f'Fetching building footprints for: {STUDY_AREA} ...')

boundary = ox.geocode_to_gdf(STUDY_AREA)
boundary_wm = boundary.to_crs(CRS_WEB_MERCATOR)
boundary_poly = boundary.geometry.iloc[0]

buildings_raw = ox.features_from_place(STUDY_AREA, tags=BUILDING_TAGS)
buildings = buildings_raw[
    buildings_raw.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])
].copy()
buildings = buildings[buildings.geometry.within(boundary_poly)].copy()

print(f'    Raw building footprints fetched: {len(buildings):,}')

## 9.2 & 9.3 -- Multi-Tier Height Engine & 3D Volumetric Calculations

Height estimation fallback hierarchy:
1. `building:levels` $\times 3.5\text{m}$
2. Explicit `height` tag
3. OSM Building type heuristics (Commercial/Hospital vs Residential/Hut)
4. Logarithmic footprint area scaling proxy: $H_{proxy} = H_{base} + 0.8 \cdot \ln(1 + A)$

**Volumetric Formulas:**
- $V = A_{\text{footprint}} \times H_{\text{height}} \quad [\text{m}^3]$
- $GFA = A_{\text{footprint}} \times \lfloor H / 3.5 \rfloor \quad [\text{m}^2]$

In [ ]:
buildings_utm = buildings.to_crs(CRS_UTM_ACCRA)
buildings['area_m2'] = buildings_utm.geometry.area.round(1)

def estimate_height(row):
    levels = row.get('building:levels')
    try:
        if pd.notna(levels) and float(levels) > 0:
            return float(levels) * 3.5
    except Exception:
        pass

    height = row.get('height')
    try:
        if pd.notna(height):
            clean_h = float(str(height).replace('m', '').replace('ft', '').strip())
            if clean_h > 0:
                return clean_h
    except Exception:
        pass

    btype = str(row.get('building', '')).lower()
    if btype in ['apartments', 'commercial', 'office', 'hospital', 'university']:
        base_h = 12.0
    elif btype in ['industrial', 'warehouse', 'retail']:
        base_h = 7.0
    elif btype in ['house', 'residential', 'detached', 'hut', 'kiosk']:
        base_h = 4.0
    else:
        base_h = 4.5

    area = row['area_m2']
    area_boost = float(np.log1p(area) * 0.8)
    return float(np.clip(base_h + area_boost, 3.5, 90.0))

buildings['height_m'] = buildings.apply(estimate_height, axis=1)
buildings['floors'] = np.maximum(1, np.floor(buildings['height_m'] / 3.5)).astype(int)
buildings['volume_m3'] = buildings['area_m2'] * buildings['height_m']
buildings['gfa_m2'] = buildings['area_m2'] * buildings['floors']

print(f'    Height Statistics (m)   : Min={buildings["height_m"].min():.1f}, Max={buildings["height_m"].max():.1f}, Mean={buildings["height_m"].mean():.1f}')
print(f'    Total Built Volume (m^3) : {buildings["volume_m3"].sum():,.0f}')
print(f'    Total Gross Floor Area  : {buildings["gfa_m2"].sum():,.0f} m^2')

## 9.4 -- H3 Hexagonal FAR & Volumetric Density Aggregation

Integrating H3 Resolution 9 discrete cells (~0.1 km$^2$ per cell) to compute **Floor Area Ratio ($FAR$)**:

$$FAR = \frac{\sum GFA_i}{A_{\text{cell}}}$$

In [ ]:
H3_RES = 9
buildings['centroid'] = buildings.geometry.centroid

bldg_h3 = [
    h3.latlng_to_cell(lat, lng, H3_RES)
    for lat, lng in zip(buildings['centroid'].y, buildings['centroid'].x)
]
buildings['h3_index'] = bldg_h3

hex_vol = buildings.groupby('h3_index').agg(
    bldg_count=('area_m2', 'count'),
    total_footprint_m2=('area_m2', 'sum'),
    total_volume_m3=('volume_m3', 'sum'),
    total_gfa_m2=('gfa_m2', 'sum'),
    mean_height_m=('height_m', 'mean'),
    max_height_m=('height_m', 'max'),
).reset_index()

hex_vol['geometry'] = hex_vol['h3_index'].apply(h3_to_polygon)
hex_gdf = gpd.GeoDataFrame(hex_vol, geometry='geometry', crs='EPSG:4326')

cell_area_m2 = h3.cell_area(list(hex_gdf['h3_index'])[0], unit='m^2')
hex_gdf['far'] = hex_gdf['total_gfa_m2'] / cell_area_m2
hex_gdf['bcr'] = (hex_gdf['total_footprint_m2'] / cell_area_m2).clip(0, 1)
hex_gdf_wm = hex_gdf.to_crs(CRS_WEB_MERCATOR)

print(f'    Hexagons covered    : {len(hex_gdf):,}')
print(f'    Max FAR observed    : {hex_gdf["far"].max():.2f}')
print(f'    Mean FAR across city: {hex_gdf["far"].mean():.3f}')

## 9.5 -- Statistical Height Distribution & Volumetric Metrics

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
fig.patch.set_facecolor(DARK_BG)

ax1.set_facecolor(DARK_BG)
ax1.hist(buildings['height_m'], bins=45, color='#ff9800', edgecolor='#111', alpha=0.85)
ax1.axvline(buildings['height_m'].mean(), color='white', linestyle='--', linewidth=1.5, label=f'Mean ({buildings["height_m"].mean():.1f}m)')
ax1.axvline(buildings['height_m'].median(), color='#00e5ff', linestyle=':', linewidth=1.5, label=f'Median ({buildings["height_m"].median():.1f}m)')
ax1.set_title(f'Building Height Distribution -- {STUDY_AREA}', fontsize=14, fontweight='bold', color='white', pad=15)
ax1.set_xlabel('Estimated Height (meters)', color='white', fontsize=11)
ax1.set_ylabel('Building Footprint Count', color='white', fontsize=11)
ax1.tick_params(colors='white')
ax1.grid(True, linestyle='--', alpha=0.2, color='white')
ax1.legend(loc='upper right', facecolor='#1a1a2e', edgecolor='white', labelcolor='white')

ax2.set_facecolor(DARK_BG)
scatter = ax2.scatter(
    buildings['area_m2'], buildings['height_m'],
    c=buildings['volume_m3'], cmap='YlOrRd', s=8, alpha=0.6,
)
ax2.set_xscale('log')
ax2.set_title('Footprint Area vs. Height (Color = Volume)', fontsize=14, fontweight='bold', color='white', pad=15)
ax2.set_xlabel('Footprint Area (m^2, log scale)', color='white', fontsize=11)
ax2.set_ylabel('Height (meters)', color='white', fontsize=11)
ax2.tick_params(colors='white')
ax2.grid(True, linestyle='--', alpha=0.2, color='white')
cbar = plt.colorbar(scatter, ax=ax2)
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(plt.getp(cbar.ax, 'yticklabels'), color='white')
cbar.set_label('Volume (m^3)', color='white')

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'ch9_building_heights.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()

## 9.6 -- Dual 2D Spatial Comparison (Footprint Density vs 3D Volume)

Comparing 2D footprint area (m$^2$) vs 3D gross built volume (m$^3$). High footprint + low volume = single-story compound sprawl; high footprint + high volume = major vertical activity center.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(28, 14))
fig.patch.set_facecolor(DARK_BG)

for ax, col, cmap, title in [
    (ax1, 'total_footprint_m2', 'YlGnBu', '2D Building Footprint Area (m^2)'),
    (ax2, 'total_volume_m3', 'YlOrRd', '3D Gross Building Volume (m^3)'),
]:
    ax.set_facecolor(DARK_BG)
    hex_gdf_wm.plot(
        ax=ax, column=col, cmap=cmap,
        scheme='natural_breaks', k=7,
        linewidth=0.1, edgecolor='#222', alpha=0.85,
        legend=True,
        legend_kwds={
            'loc': 'lower left', 'fontsize': 8,
            'facecolor': '#1a1a2e', 'edgecolor': 'white',
            'labelcolor': 'white', 'framealpha': 0.9,
            'title': title})
    boundary_wm.plot(ax=ax, facecolor='none', edgecolor='white', linewidth=2, zorder=10)
    try:
        cx.add_basemap(ax, source=cx.providers.CartoDB.DarkMatter, zoom=12)
    except Exception:
        pass
    ax.set_axis_off()
    ax.set_title(title, fontsize=15, fontweight='bold', color='white', pad=12)

fig.suptitle(f'2D FOOTPRINT DENSITY vs 3D VOLUMETRIC DENSITY -- {STUDY_AREA}',
             fontsize=18, fontweight='bold', color='white', y=0.97)
fig.tight_layout(rect=[0, 0, 1, 0.94])
fig.savefig(FIGURES_DIR / 'ch9_footprint_vs_volume_comparison.png',
            dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()

## 9.7 -- District-Level Volumetric Ranking & Skyline Profiles

In [ ]:
try:
    district_names = [
        'Accra Metropolitan District, Greater Accra, Ghana',
        'Ablekuma Central Municipal District, Greater Accra, Ghana',
        'Ablekuma West Municipal District, Greater Accra, Ghana',
        'Ayawaso Central Municipal District, Greater Accra, Ghana',
        'Ayawaso East Municipal District, Greater Accra, Ghana',
        'Ayawaso West Municipal District, Greater Accra, Ghana',
        'Korle Klottey Municipal District, Greater Accra, Ghana',
        'La Dade-Kotopon Municipal District, Greater Accra, Ghana',
        'Okaikwei North Municipal District, Greater Accra, Ghana',
        'Tema Metropolitan District, Greater Accra, Ghana',
    ]
    districts_gdf = ox.geocode_to_gdf(district_names)
    bldgs_with_dist = gpd.sjoin(
        buildings.to_crs(districts_gdf.crs), districts_gdf[['display_name', 'geometry']],
        how='inner', predicate='within',
    )
    bldgs_with_dist['district_clean'] = bldgs_with_dist['display_name'].apply(
        lambda s: str(s).split(',')[0].replace('Municipal District', '').replace('Metropolitan District', '').strip()
    )

    dist_stats = bldgs_with_dist.groupby('district_clean').agg(
        total_vol_m3=('volume_m3', 'sum'),
        p95_height=('height_m', lambda x: np.percentile(x, 95)),
        mean_height=('height_m', 'mean'),
        count=('area_m2', 'count'),
    ).reset_index().sort_values(by='total_vol_m3', ascending=False)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))
    fig.patch.set_facecolor(DARK_BG)

    ax1.set_facecolor(DARK_BG)
    ax1.barh(dist_stats['district_clean'], dist_stats['total_vol_m3'] / 1e6, color='#ff9800', edgecolor='white', linewidth=0.8)
    ax1.set_xlabel('Total Built Volume (Million m^3)', color='white', fontsize=11)
    ax1.set_title('District Ranking by Total Built Volume', fontsize=14, fontweight='bold', color='white', pad=12)
    ax1.tick_params(colors='white')
    ax1.grid(True, axis='x', linestyle='--', alpha=0.2, color='white')
    ax1.invert_yaxis()

    ax2.set_facecolor(DARK_BG)
    ax2.barh(dist_stats['district_clean'], dist_stats['p95_height'], color='#00e5ff', edgecolor='white', linewidth=0.8)
    ax2.set_xlabel('95th Percentile Building Height (m)', color='white', fontsize=11)
    ax2.set_title('District Ranking by Peak Skyline Height', fontsize=14, fontweight='bold', color='white', pad=12)
    ax2.tick_params(colors='white')
    ax2.grid(True, axis='x', linestyle='--', alpha=0.2, color='white')
    ax2.invert_yaxis()

    fig.suptitle(f'DISTRICT SKYLINE & VOLUMETRIC PROFILES -- {STUDY_AREA}', fontsize=16, fontweight='bold', color='white', y=0.98)
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    fig.savefig(FIGURES_DIR / 'ch9_district_skyline_rankings.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
    plt.show()
except Exception as e:
    print(f'District spatial join skipped: {e}')

## 9.8 -- Interactive 3D Pydeck Cityscape (Extruded Polygons)

Interactive WebGL rendering with pitch=60$^\circ$ and height-based color ramp.

> **Performance Optimization:** Filters out micro-sheds ($<15\text{m}^2$) and caps at the top 25,000 prominent buildings by volume to ensure high-speed, crash-free 3D WebGL rendering.

In [ ]:
print('Rendering Interactive 3D Pydeck Cityscape ...')

q99_area = buildings['area_m2'].quantile(0.99)
bldg_filtered = buildings[
    (buildings['area_m2'] >= 15.0) & (buildings['area_m2'] <= q99_area)
].copy()

if len(bldg_filtered) > 25000:
    print(f'    [Optimization] Capping {len(bldg_filtered):,} buildings to top 25,000 by volume for smooth 3D WebGL rendering ...')
    bldg_filtered = bldg_filtered.sort_values(by='volume_m3', ascending=False).head(25000)

bldg_filtered = bldg_filtered.to_crs('EPSG:4326').copy()

def get_coordinates(geom):
    if geom.geom_type == 'Polygon':
        return [list(coord)[:2] for coord in geom.exterior.coords]
    elif geom.geom_type == 'MultiPolygon':
        return [list(coord)[:2] for coord in list(geom.geoms)[0].exterior.coords]
    return []

bldg_filtered['polygon'] = bldg_filtered.geometry.apply(get_coordinates)
bldg_filtered = bldg_filtered[bldg_filtered['polygon'].apply(len) > 0].copy()

max_h = bldg_filtered['height_m'].quantile(0.98) or 1
bldg_filtered['norm_h'] = (bldg_filtered['height_m'] / max_h).clip(0, 1)

bldg_filtered['r'] = (255 * bldg_filtered['norm_h']).astype(int)
bldg_filtered['g'] = (180 * (1 - bldg_filtered['norm_h'])).astype(int)
bldg_filtered['b'] = (255 * (1 - bldg_filtered['norm_h'])).astype(int)
bldg_filtered['color'] = bldg_filtered.apply(
    lambda row: [int(row['r']), int(row['g']), int(row['b']), 220], axis=1
)

view_state = pdk.ViewState(
    latitude=5.5600, longitude=-0.2057, zoom=14, pitch=60, bearing=-20
)

layer_3d_skyline = pdk.Layer(
    'PolygonLayer',
    bldg_filtered,
    get_polygon='polygon',
    get_elevation='height_m',
    get_fill_color='color',
    extruded=True,
    wireframe=True,
    get_line_color=[30, 30, 30, 100],
    line_width_min_pixels=1,
    pickable=True,
    auto_highlight=True,
)

deck_skyline = pdk.Deck(
    layers=[layer_3d_skyline],
    initial_view_state=view_state,
    map_style=pdk.map_styles.CARTO_DARK,
    tooltip={
        'text': 'Building Height: {height_m}m\nEstimated Floors: {floors}\nFootprint Area: {area_m2} m^2\nGross Volume: {volume_m3} m^3'
    },
)

html_skyline = OUTPUT_DIR / 'ch9_3d_accra_skyline.html'
deck_skyline.to_html(str(html_skyline))
print(f'Saved: {html_skyline}')

## 9.9 -- Interactive Dual-Metric 3D Deck (Height=H, Color=FAR)

3D Extrusion height represents maximum building height ($H_{max}$), while color represents Floor Area Ratio ($FAR$).

In [ ]:
hex_dual_3d = hex_gdf.copy()
hex_dual_3d['elevation'] = hex_dual_3d['max_height_m'] * 15

max_far = hex_dual_3d['far'].quantile(0.98) or 1
hex_dual_3d['r'] = (255 * (hex_dual_3d['far'] / max_far).clip(0, 1)).astype(int)
hex_dual_3d['g'] = (140 * (1 - hex_dual_3d['far'] / max_far).clip(0, 1)).astype(int)
hex_dual_3d['b'] = (255 * (1 - hex_dual_3d['far'] / max_far).clip(0, 1)).astype(int)

layer_far_3d = pdk.Layer(
    'H3HexagonLayer',
    hex_dual_3d,
    get_hexagon='h3_index',
    get_fill_color='[r, g, b, 210]',
    get_elevation='elevation',
    elevation_scale=1,
    extruded=True,
    pickable=True,
    auto_highlight=True,
)

deck_far_3d = pdk.Deck(
    layers=[layer_far_3d],
    initial_view_state=view_state,
    map_style=pdk.map_styles.CARTO_DARK,
    tooltip={
        'text': 'H3 Cell: {h3_index}\nMax Building Height: {max_height_m}m\nFloor Area Ratio (FAR): {far}\nTotal Built Volume: {total_volume_m3} m^3'
    },
)

html_far = OUTPUT_DIR / 'ch9_3d_far_volumetric.html'
deck_far_3d.to_html(str(html_far))
print(f'Saved: {html_far}')

---

## Professor Innovation 1: Sky View Factor (SVF) & Urban Canyon Proxy

**Sky View Factor ($SVF$)** (Oke, 1988) measures the proportion of visible sky from street level:

$$SVF \approx \cos\left(\arctan\left(\frac{2 \bar{H}}{W_{\text{street}}}\right)\right)$$

where $\bar{H}$ is mean building height per hexagon, and $W_{\text{street}} \approx 15.0\text{ m}$ is average street width.
- $SVF \to 0.0$: Deep urban canyons with trapped longwave radiation (UHI risk)
- $SVF \to 1.0$: Open, low-density rural or suburban fabric

> *Reference: Oke, T. R. (1988). Street Design and Urban Canopy Layer Climate.*

In [ ]:
STREET_WIDTH_M = 15.0
hex_gdf['svf_proxy'] = np.cos(
    np.arctan(2.0 * hex_gdf['mean_height_m'] / STREET_WIDTH_M)
).clip(0.1, 1.0)
hex_gdf_wm['svf_proxy'] = hex_gdf['svf_proxy']

print(f'SVF Statistics : Min={hex_gdf["svf_proxy"].min():.3f}, Max={hex_gdf["svf_proxy"].max():.3f}, Mean={hex_gdf["svf_proxy"].mean():.3f}')

fig, ax = plt.subplots(figsize=(18, 18))
fig.patch.set_facecolor(DARK_BG)
ax.set_facecolor(DARK_BG)

hex_gdf_wm.plot(
    ax=ax, column='svf_proxy', cmap='viridis_r',
    scheme='natural_breaks', k=7,
    linewidth=0.1, edgecolor='#222', alpha=0.85,
    legend=True,
    legend_kwds={
        'loc': 'lower left', 'fontsize': 9,
        'facecolor': '#1a1a2e', 'edgecolor': 'white',
        'labelcolor': 'white', 'framealpha': 0.9,
        'title': 'Sky View Factor (SVF)'})

boundary_wm.plot(ax=ax, facecolor='none', edgecolor='white', linewidth=2.5, zorder=10)

try:
    cx.add_basemap(ax, source=cx.providers.CartoDB.DarkMatter, zoom=13)
except Exception:
    pass

ax.set_axis_off()
ax.set_title(
    'PROFESSOR\'S INNOVATION 1 -- Sky View Factor (SVF) Urban Canyon Proxy\n'
    'Lower SVF (Dark) = High Urban Canyon Enclosure | Higher SVF (Bright) = Open Sky Access',
    fontsize=16, fontweight='bold', color='white', pad=20)

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'ch9_professor_sky_view_factor.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()

## Professor Innovation 2: Spatial Floor Area Ratio (FAR) & Sprawl Index

**Floor Area Ratio ($FAR$)** (Leslie & Martin, 1972) measures built intensity per land unit:

$$FAR = \frac{\sum \text{Gross Floor Area}_i}{A_{\text{cell}}}$$

- $FAR > 2.0$: High-density vertical urban compaction
- $FAR < 0.3$: Low-density horizontal sprawl

> *Reference: Leslie, M. & Martin, L. (1972). Urban Space and Structures.*

In [ ]:
fig, ax = plt.subplots(figsize=(18, 18))
fig.patch.set_facecolor(DARK_BG)
ax.set_facecolor(DARK_BG)

hex_gdf_wm.plot(
    ax=ax, column='far', cmap='magma',
    scheme='natural_breaks', k=7,
    linewidth=0.1, edgecolor='#222', alpha=0.85,
    legend=True,
    legend_kwds={
        'loc': 'lower left', 'fontsize': 9,
        'facecolor': '#1a1a2e', 'edgecolor': 'white',
        'labelcolor': 'white', 'framealpha': 0.9,
        'title': 'Floor Area Ratio (FAR)'})

boundary_wm.plot(ax=ax, facecolor='none', edgecolor='white', linewidth=2.5, zorder=10)

try:
    cx.add_basemap(ax, source=cx.providers.CartoDB.DarkMatter, zoom=13)
except Exception:
    pass

ax.set_axis_off()
ax.set_title(
    'PROFESSOR\'S INNOVATION 2 -- Spatial Floor Area Ratio (FAR) & Urban Intensity\n'
    'Quantifying Vertical Compaction (High FAR) vs. Horizontal Sprawl (Low FAR)',
    fontsize=16, fontweight='bold', color='white', pad=20)

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'ch9_professor_spatial_far.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()

## Professor Innovation 3: Volumetric Thermal Mass & UHI Vulnerability

**Thermal Mass Index ($TMI$)** (Voogt & Oke, 2003) measures total thermal mass (concrete volume) per land area:

$$TMI = \frac{\sum V_{\text{building}}}{A_{\text{cell}}} \quad [\text{m}]$$

Higher thermal mass absorbs heat during the day and re-radiates it at night, creating **Urban Heat Island (UHI)** microclimates.

> *Reference: Voogt, J. A., & Oke, T. R. (2003). Thermal Remote Sensing of Urban Climates.*

In [ ]:
hex_gdf['tmi_m'] = hex_gdf['total_volume_m3'] / cell_area_m2
hex_gdf_wm['tmi_m'] = hex_gdf['tmi_m']

print(f'Thermal Mass Index (m) : Min={hex_gdf["tmi_m"].min():.2f}, Max={hex_gdf["tmi_m"].max():.2f}, Mean={hex_gdf["tmi_m"].mean():.2f}')

fig, ax = plt.subplots(figsize=(18, 18))
fig.patch.set_facecolor(DARK_BG)
ax.set_facecolor(DARK_BG)

hex_gdf_wm.plot(
    ax=ax, column='tmi_m', cmap='inferno',
    scheme='natural_breaks', k=7,
    linewidth=0.1, edgecolor='#222', alpha=0.85,
    legend=True,
    legend_kwds={
        'loc': 'lower left', 'fontsize': 9,
        'facecolor': '#1a1a2e', 'edgecolor': 'white',
        'labelcolor': 'white', 'framealpha': 0.9,
        'title': 'Thermal Mass Index (V / A)'})

boundary_wm.plot(ax=ax, facecolor='none', edgecolor='white', linewidth=2.5, zorder=10)

try:
    cx.add_basemap(ax, source=cx.providers.CartoDB.DarkMatter, zoom=13)
except Exception:
    pass

ax.set_axis_off()
ax.set_title(
    'PROFESSOR\'S INNOVATION 3 -- Volumetric Thermal Mass & UHI Vulnerability\n'
    'Higher TMI (Bright Yellow) = Extreme Concrete Mass & Nocturnal Heat Retention Risk',
    fontsize=16, fontweight='bold', color='white', pad=20)

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'ch9_professor_thermal_mass_uhi.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()

---

## Chapter 9 Complete

### Summary

| Section | Method | Key Insight |
|---------|--------|------------|
| 9.1-9.3 | Volumetric Engine | $V = A \times H$, $GFA = A \times \text{Floors}$ |
| 9.4 | H3 Res 9 FAR | Spatial distribution of built intensity |
| 9.6 | Footprint vs Volume | Footprint density $\neq$ Volumetric density |
| 9.8-9.9 | WebGL 3D Pydeck | Interactive 3D skylines and dual-metric mapping |
| Innovation 1 | Sky View Factor (SVF) | Street-level solar access & canyon enclosure |
| Innovation 2 | Floor Area Ratio (FAR) | Vertical compaction vs horizontal sprawl |
| Innovation 3 | Thermal Mass Index | Urban Heat Island (UHI) vulnerability proxy |

### Academic References
- Oke, T. R. (1988). *Street Design and Urban Canopy Layer Climate.* Building and Environment.
- Leslie, M. & Martin, L. (1972). *Urban Space and Structures.* Cambridge University Press.
- Voogt, J. A., & Oke, T. R. (2003). *Thermal remote sensing of urban climates.* Remote Sensing of Environment.